<a href="https://colab.research.google.com/github/Lasisi-Samson/YORI_PIPELINE/blob/main/COMPLETE_YORI_PIPELINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install intel_extension_for_pytorch


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!pip install --upgrade autoawq

In [ ]:
import torch
import torch.nn as nn
import transformers.activations as activations

# Patch transformers.activations if PytorchGELUTanh is missing
if not hasattr(activations, "PytorchGELUTanh"):
    class PytorchGELUTanh(nn.Module):
        def forward(self, x):
            return 0.5 * x * (1.0 + torch.tanh((2.0 / torch.pi)**0.5 * (x + 0.044715 * x**3)))
    activations.PytorchGELUTanh = PytorchGELUTanh

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer


/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

ERROR! Intel® Extension for PyTorch* needs to work with PyTorch 2.8.*, but PyTorch 2.9.0+cpu is found. Please switch to the matching version and run again.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


ERROR! Intel® Extension for PyTorch* needs to work with PyTorch 2.8.*, but PyTorch 2.9.0+cpu is found. Please switch to the matching version and run again.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
!pip install gptqmodel

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
import torch, os

model_id = "Lasisi/YORI-Llama-Quantized"
output_dir = "./YORI-Llama-FP16"

# Load tokenizer + AWQ model on GPU
tokenizer = AutoTokenizer.from_pretrained(model_id)
awq_model = AutoAWQForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

# Get the base LLaMA model (FP16)
base_model = awq_model.model

# Save in Hugging Face FP16 format
os.makedirs(output_dir, exist_ok=True)
base_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Exported to {output_dir} in FP16 format")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

Loading weights:   0%|          | 0/739 [00:00<?, ?it/s]

INFO  Optimize: `TorchAtenAwqLinear` compilation triggered.                    


INFO  gc.collect() reclaimed 6394 objects in 0.239s                            


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Exported to ./YORI-Llama-FP16 in FP16 format


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_dir = "./YORI-Llama-FP16"  # your exported FP16 model path

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# Load FP16 model
device = "cuda" if torch.cuda.is_available() else "cpu"
device_map = "cuda"
model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.float16).to(device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


INFO  Kernel: Auto-selection: adding candidate `TorchAtenAwqLinear`            


INFO  Kernel: selected -> `TorchAtenAwqLinear`.                                


Loading weights:   0%|          | 0/739 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6488 objects in 0.241s                            


In [ ]:
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="AstralZander/yoruba_ASR")
from transformers import AutoProcessor, AutoModelForCTC

processor = AutoProcessor.from_pretrained("AstralZander/yoruba_ASR")
model_1 = AutoModelForCTC.from_pretrained("AstralZander/yoruba_ASR")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

In [ ]:
# @title
# ⬇️ Install necessary library
!pip install soundfile

# 📸 Record Audio in Colab
import IPython
from IPython.display import Audio, display, Javascript
from google.colab import output
import soundfile as sf
import numpy as np
import io
import base64

# ✅ JavaScript for recording audio in Colab
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = () => resolve(reader.result);
  reader.readAsDataURL(blob);
});

var record = async function(duration=5000) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();

  await sleep(duration);
  recorder.stop();

  await new Promise(resolve => recorder.onstop = resolve);
  const blob = new Blob(chunks);
  return await b2text(blob);
}
"""

def record_audio(filename="recorded.wav", duration=5):
    print(f"🎙️ Recording for {duration} seconds...")

    display(Javascript(RECORD_JS))
    data = output.eval_js(f"record({duration * 1000})")  # duration in ms

    # Decode base64 audio
    header, encoded = data.split(",", 1)
    audio_bytes = base64.b64decode(encoded)

    # Save to WAV file
    with open(filename, "wb") as f:
        f.write(audio_bytes)

    print(f"✅ Saved as: {filename}")
    return filename

In [ ]:
# @title
# 🔴 Record audio
recorded_file = record_audio("yoruba.wav", duration=3)

# 🎧 Playback
Audio("yoruba.wav")

🎙️ Recording for 3 seconds...


<IPython.core.display.Javascript object>

✅ Saved as: yoruba.wav


In [ ]:
# @title
audio_path = "yoruba.wav"

# Run inference
output = pipe(audio_path)

transcribed_text = output["text"]
print("🔊 Transcription:", transcribed_text)

🔊 Transcription: Ni orúkọ rẹ.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# @title
user_input = transcribed_text
print (f"User: {user_input}")

User: Ni orúkọ rẹ.


In [ ]:
# @title
prompt = f"""
You are Yori, a helpful assistant that only speaks Yorùbá professionally  .

Q: {transcribed_text}
A:
"""



In [ ]:
# @title


outputs = model.generate(
    **tokenizer(prompt, return_tensors="pt").to(model.device),
    max_new_tokens= 70,
    temperature=0.9,
    top_p=0.95,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,   # stop at end-of-sequence
    pad_token_id=tokenizer.eos_token_id,

)

reply = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Remove the user prompt if echoed
reply = reply.replace(prompt, "").strip()
# Keep only the part after "A:"
if "A:" in reply:
  reply = reply.split("A:")[1].strip()

# Print or return the reply
print("🤖 Yori:", reply)

[transformers] Both `max_new_tokens` (=70) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


🤖 Yori: Ní àkókò òfíìdí mi, mo fẹ


In [ ]:
# @title
print(reply)

NameError: name 'reply' is not defined

In [ ]:
# @title
from transformers import pipeline
import numpy as np
from scipy.io.wavfile import write

# Load Yoruba TTS pipeline
pipe = pipeline("text-to-speech", model="facebook/mms-tts-yor", device=0)

# Longer Yoruba text
text = '''E kaaro! Bawo ni mo se le ran yin lowo?'''


# Split by sentence
sentences = [s.strip() for s in text.split("\n") if s.strip()]

all_audio = []
sr = None  # sampling rate

for sentence in sentences:
    output = pipe(sentence)
    audio = np.array(output["audio"] * 32767, dtype=np.int16).flatten()  # ✅ flatten
    all_audio.append(audio)
    sr = int(output["sampling_rate"])  # ✅ ensure Python int

# Concatenate all chunks
final_audio = np.concatenate(all_audio, axis=0)

# Save as WAV
write("mms_yoruba_long.wav", sr, final_audio)

print("✅ Saved to mms_yoruba_long.wav")

In [ ]:
# @title
from IPython.display import Audio

# If your output is a numpy array + sampling rate
audio_array = output["audio"]
sampling_rate = output["sampling_rate"]

# Play directly
Audio(audio_array, rate=sampling_rate)